### 01. Import Dependecies

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,OneHotEncoder,OrdinalEncoder,LabelEncoder
)

### 02. Loading Data

In [2]:
df = pd.read_csv("data/processed/estate_Binning_Applied.csv")
df.head(5)

,IncomeLevel,TotalRooms,TotalBedrooms,NeighborhoodPop,AvgOccupancy,Latitude,Longitude,TargetPrice,RoomsPerHousehold,BedroomsRatio,PropertyAge_bins
0,3.287977,5.128150,0.990769,2339.474039,3.739113,32.71,-117.03,1.030,1.359130,0.200576,Moderate
1,3.804601,4.372696,1.040469,1269.383596,1.429576,33.77,-118.16,3.821,2.573820,0.232703,Old
2,2.029509,4.131418,1.032285,1455.381923,3.914447,32.69,-117.11,0.934,1.002116,0.258269,Old
3,3.540823,6.270531,1.147146,895.050628,2.681969,36.78,-119.80,0.965,2.725400,0.180940,Old
4,6.609324,6.178103,0.986182,2698.088189,3.457448,37.42,-121.86,2.648,1.867161,0.160572,Moderate


### 03. Data Preprocessing

In [3]:
ordinal_features = ["PropertyAge_bins"]
nominal_features = []
numerical_features = [
    "IncomeLevel", "TotalRooms", "TotalBedrooms", "NeighborhoodPop",
    "AvgOccupancy", "Latitude", "Longitude", "RoomsPerHousehold", "BedroomsRatio"
]
remainder_features = []
target_feature = "TargetPrice"

#### 3.1 Data Splitting

In [4]:
X = df.drop(columns=[target_feature])
y = df[target_feature]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

#### 3.2 Scaling and Encoding

In [6]:
numerical_transformer = Pipeline(
                                steps=[
                                    ('imputer', SimpleImputer(strategy='median')),
                                    ('scaler', StandardScaler())
                                ]
                                )

norminal_transformer = Pipeline(
                                steps=[
                                    ('imputer', SimpleImputer(
                                                            strategy='constant',
                                                            fill_value='missing'
                                                            )),
                                    ('encoder', OneHotEncoder())
                                ]
                                )

ordinal_transformer = Pipeline(
                                steps=[
                                    ('imputer', SimpleImputer(
                                                            strategy='constant',
                                                            fill_value='missing'
                                                            )),
                                    ('encoder', OrdinalEncoder())
                                ]
                                )

preprocessor = ColumnTransformer(
    transformers=[
        ('num',numerical_transformer,numerical_features),
        ('nom',norminal_transformer,nominal_features),
        ('ord',ordinal_transformer,ordinal_features)
    ],
    remainder='passthrough'
)

nominal_features_name=[]
for feature in nominal_features:
    unique_values=df[feature].unique()
    nominal_features_name.extend([f"{feature}_{val}" for val in unique_values])

In [7]:
X_train = pd.DataFrame(
    preprocessor.fit_transform(X_train,y_train),
    columns=numerical_features+nominal_features_name+ordinal_features+remainder_features
)

In [8]:
X_test = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=numerical_features+nominal_features_name+ordinal_features+remainder_features
)

In [9]:
y_train = pd.DataFrame(y_train).reset_index(drop=True)
y_test = pd.DataFrame(y_test).reset_index(drop=True)

In [10]:
print(y_train['TargetPrice'].value_counts()/len(y_train) * 100)

TargetPrice
5.00001    4.770129
1.62500    0.583930
1.37500    0.534583
1.12500    0.485237
1.87500    0.427667
             ...   
4.57800    0.008224
3.39700    0.008224
4.80800    0.008224
3.44300    0.008224
3.53500    0.008224
Name: count, Length: 3418, dtype: float64


### 4. Saving Files

In [11]:
X_train.to_csv("artifacts/X_train.csv",index=False)
y_train.to_csv("artifacts/y_train.csv",index=False)
X_test.to_csv("artifacts/X_test.csv",index=False)
y_test.to_csv("artifacts/y_test.csv",index=False)